In [4]:
import json

with open("../data/match.json") as f:
    data = json.load(f)

data

[{'home team': 'Manchester United',
  'away team': 'Brighton',
  'score': '4-2',
  'goals': [{'minute': '24',
    'player': 'Matheus Cunha',
    'team': 'Manchester United'},
   {'minute': '34', 'player': 'Casemiro', 'team': 'Manchester United'},
   {'minute': '61', 'player': 'Bryan Mbeumo', 'team': 'Manchester United'},
   {'minute': '96', 'player': 'Bryan Mbeumo', 'team': 'Manchester United'},
   {'minute': '74', 'player': 'Danny Welbeck', 'team': 'Brighton'},
   {'minute': '96', 'player': 'Charalampos Kostoulas', 'team': 'Brighton'}],
  'red cards': [],
  'home manager': 'Ruben Amorim',
  'away manager': 'Fabian Hurzeler',
  'man of the match': '',
  'stadium': 'Old Trafford'},
 {'home team': 'Manchester United',
  'away team': 'Bournemouth',
  'score': '4-4',
  'goals': [{'minute': '13', 'player': 'Semenyo', 'team': 'Bournemouth'},
   {'minute': '40', 'player': 'Bruno Fernandes', 'team': 'Manchester United'},
   {'minute': '44', 'player': 'Matheus Cunha', 'team': 'Manchester United

In [5]:
# Code that produces first Excel spreadsheet (match level data)

import pandas as pd

match_rows = []
for match in data:
    match_rows.append({
        "home team": match["home team"],
        "home manager": match["home manager"],
        "home score": match["score"].split("-")[0],
        "away team": match["away team"],
        "away manager": match["away manager"],
        "away score": match["score"].split("-")[1],
    })
matches_df = pd.DataFrame(match_rows)
matches_df

,home team,home manager,home score,away team,away manager,away score
0,Manchester United,Ruben Amorim,4,Brighton,Fabian Hurzeler,2
1,Manchester United,Ruben Amorim,4,Bournemouth,Andoni Iraola,4


In [6]:
def summarise_goals(goals):
    return ", ".join([
        f"{g['player']} ({g['minute']})"
        for g in goals
    ])

matches_df["goals_summary"] = [
    summarise_goals(match.get("goals", []))
    for match in data
]

In [7]:
matches_df

,home team,home manager,home score,away team,away manager,away score,goals_summary
0,Manchester United,Ruben Amorim,4,Brighton,Fabian Hurzeler,2,"Matheus Cunha (24), Casemiro (34), Bryan Mbeum..."
1,Manchester United,Ruben Amorim,4,Bournemouth,Andoni Iraola,4,"Semenyo (13), Bruno Fernandes (40), Matheus Cu..."


In [18]:
from collections import Counter

goal_scorers = []
for match in data:
    for goal in match.get("goals", []):
        goal_scorers.append(goal["player"])

counts = Counter(goal_scorers)

ranking_df = (
    pd.DataFrame(
        counts.items(),
        columns=["Player", "Goals Seen Live"]
    )
    .sort_values(
        ["Goals Seen Live", "Player"],
        ascending=[False, True]
    )
)

ranking_df["Rank"] = (
    ranking_df["Goals Seen Live"]
    .rank(method="min", ascending=False)
    .astype(int)
)

ranking_df = ranking_df[
    ["Rank", "Player", "Goals Seen Live"]
].reset_index(drop=True)

ranking_df

,Rank,Player,Goals Seen Live
0,1,Bruno Fernandes,2
1,1,Bryan Mbeumo,2
2,1,Eli Junior Kroupi,2
3,1,Matheus Cunha,2
4,5,Casemiro,1
5,5,Charalampos Kostoulas,1
6,5,Danny Welbeck,1
7,5,Evanilson,1
8,5,Marcus Tavernier,1
9,5,Semenyo,1


In [21]:
with pd.ExcelWriter("../data/enriched_matches.xlsx", engine="openpyxl") as writer:
    matches_df.to_excel(writer, sheet_name="Matches", index=False)
    ranking_df.to_excel(writer, sheet_name="Top Scorers", index=False)